<a href="https://colab.research.google.com/github/Sangeetha231005/MLOps_Training/blob/main/Copy_of_Disaster_Detection_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Disaster Detection using YOLOv11**

In [ ]:
!pip install ultralytics
!pip install opencv-python

In [ ]:
import json
import os
import shutil
import random
from pathlib import Path

victim_dir = "/content/drive/MyDrive/Disaster_detection/victim_frames"

novictim_dir = "/content/drive/MyDrive/Disaster_detection/novictim_frames"

output_dataset = "/content/drive/MyDrive/Disaster_detection/victim_dataset"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(output_dataset, folder), exist_ok=True)

victim_images = []
novictim_images = []

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    victim_images.extend(Path(victim_dir).glob(ext))

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    novictim_images.extend(Path(novictim_dir).glob(ext))

all_images = []

for img in victim_images:
    all_images.append(("victim", img))

for img in novictim_images:
    all_images.append(("novictim", img))

random.shuffle(all_images)

split_index = int(len(all_images) * 0.8)

train_images = all_images[:split_index]
val_images = all_images[split_index:]

def process_image(img_type, image_path, split):

    image_name = image_path.name

    dst_image = os.path.join(
        output_dataset,
        "images",
        split,
        image_name
    )

    shutil.copy(image_path, dst_image)

    txt_name = image_path.stem + ".txt"

    txt_output = os.path.join(
        output_dataset,
        "labels",
        split,
        txt_name
    )

    if img_type == "victim":

        json_name = image_path.stem + ".json"

        json_path = os.path.join(victim_dir, json_name)

        if os.path.exists(json_path):

            with open(json_path, "r") as f:
                data = json.load(f)

            image_width = data["imageWidth"]
            image_height = data["imageHeight"]

            with open(txt_output, "w") as txt_file:

                for shape in data["shapes"]:

                    if shape["shape_type"] != "polygon":
                        continue

                    class_id = 0

                    polygon = shape["points"]

                    normalized_points = []

                    for x, y in polygon:

                        x = x / image_width
                        y = y / image_height

                        normalized_points.append(str(x))
                        normalized_points.append(str(y))

                    line = str(class_id) + " " + " ".join(normalized_points)

                    txt_file.write(line + "\n")

    else:

        open(txt_output, "w").close()

for img_type, img_path in train_images:
    process_image(img_type, img_path, "train")

for img_type, img_path in val_images:
    process_image(img_type, img_path, "val")

print("Dataset preparation completed!")

In [ ]:
data_yaml = """
path: /content/drive/MyDrive/Disaster_detection/victim_dataset

train: images/train
val: images/val

names:
  0: victim
"""

yaml_path = "/content/drive/MyDrive/Disaster_detection/data.yaml"

with open(yaml_path, "w") as f:
    f.write(data_yaml)

print("data.yaml created successfully!")
print(yaml_path)

In [ ]:
from ultralytics import YOLO
import os

save_project = "/content/drive/MyDrive/Disaster_detection/training_results"

os.makedirs(save_project, exist_ok=True)

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/drive/MyDrive/Disaster_detection/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    workers=2,

    project=save_project,
    name="victim_segmentation",

    exist_ok=True,
    save=True
)

print("Training completed!")

**Test the model using video**

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/Disaster_detection/training_results/victim_segmentation/weights/best.pt"
)

results = model.predict(
    source="/content/drive/MyDrive/Disaster_detection/test.mp4",
    save=True,
    imgsz=320,
    vid_stride=10,
    device=0
)

In [ ]:
!ls -lh /content/runs/segment/predict

In [ ]:
from IPython.display import Video

Video(
    "/content/runs/segment/predict/test.avi",
    embed=True
)

**Test the model using image - victim**

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/Disaster_detection/training_results/victim_segmentation/weights/best.pt"
)

results = model.predict(
    source="/content/drive/MyDrive/Disaster_detection/victim3.jpg",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/runs/segment/predict-3

In [ ]:
from IPython.display import Image

Image(
    filename="/content/runs/segment/predict-3/victim3.jpg"
)

**Test the model using image - novictim**

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/Disaster_detection/training_results/victim_segmentation/weights/best.pt"
)

results = model.predict(
    source="/content/drive/MyDrive/Disaster_detection/test_novictim1.jpg",
    conf=0.25,
    save=True
)

In [ ]:
!ls /content/runs/segment/predict-4

In [ ]:
from IPython.display import Image

Image(
    filename="/content/runs/segment/predict-4/test_novictim1.jpg"
)